# Databricks + Azure OpenAI: Docs Agent

Build and evaluate a tool-calling documentation agent in Databricks.

**Architecture**
- **Azure OpenAI (`gpt-5.4-mini`)** provides LLM inference.
- **Databricks** runs the notebook and agent workflow.
- **LangGraph** orchestrates tool calls.
- **TF-IDF** retrieves Databricks documentation and extracts keywords.
- **Unity Catalog** hosts the keyword function when permissions allow.
- **MLflow** traces agent activity and logs evaluation results.

This notebook does not register a model or create a serving endpoint.

## 1. Install dependencies

Run this cell, wait for installation to finish, and then run the Python restart cell immediately below it. Do not run the remaining cells before the restart.

The packages are installed together so pip can resolve a compatible LangGraph dependency set. `langgraph.prebuilt` is included with the `langgraph` package; do not separately upgrade `langgraph-prebuilt` in a different cell.

In [ ]:
%pip install --upgrade \
    "mlflow[databricks]" \
    "langchain" \
    "langchain-core" \
    "langchain-openai" \
    "langgraph" \
    "databricks-langchain" \
    "unitycatalog-langchain[databricks]" \
    "scikit-learn" \
    "pandas"

In [ ]:
dbutils.library.restartPython()

## 2. Verify package versions

After the Python restart, check the installed versions and dependency consistency before importing LangGraph.

In [ ]:
from importlib.metadata import version, PackageNotFoundError
import subprocess
import sys

packages = [
    "mlflow",
    "langchain",
    "langchain-core",
    "langchain-openai",
    "langgraph",
    "langgraph-prebuilt",
    "databricks-langchain",
    "unitycatalog-langchain",
    "scikit-learn",
    "pandas",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

print("\nChecking dependency consistency...")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

## 3. Configure Azure OpenAI

Use the Azure OpenAI v1 API endpoint and the **deployment name** of your model. The deployment name must match the model deployment configured in Azure.

Enter the API key in the Databricks widget at runtime. For a shared or production workspace, prefer a Databricks secret scope.

In [ ]:
AZURE_OPENAI_BASE_URL = "https://hakunamatata11.openai.azure.com/openai/v1/"
MODEL = "gpt-5.4-mini"  # Azure deployment name

catalog = "agenticaiengineer"
schema = "default"

dbutils.widgets.text("azure_openai_key", "", "Azure OpenAI API key")
API_KEY = dbutils.widgets.get("azure_openai_key").strip()

# Recommended alternative:
# API_KEY = dbutils.secrets.get(scope="YOUR_SECRET_SCOPE", key="YOUR_SECRET_KEY")

assert API_KEY, (
    "Enter your Azure OpenAI API key in the azure_openai_key widget, "
    "or configure the Databricks secret lookup."
)

## 4. Connect to Azure OpenAI

Databricks is the execution environment. The model request is sent to Azure OpenAI using `ChatOpenAI` and the Azure OpenAI v1-compatible base URL.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    base_url=AZURE_OPENAI_BASE_URL,
    api_key=API_KEY,
    max_completion_tokens=1000,
)

connection_test = llm.invoke("Reply with one word: ready")
print(connection_test.content)

## 5. Load documentation and create tools

The document retriever uses a TF-IDF index over Databricks documentation chunks. The keyword extractor uses TF-IDF over the supplied text.

The keyword function is registered in Unity Catalog when the workspace and user permissions allow it. If registration or toolkit setup fails, the notebook uses a local LangChain tool instead.

In [ ]:
from typing import Any

import pandas as pd
import mlflow

from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer

docs_url = (
    "https://raw.githubusercontent.com/databricks/genai-cookbook/"
    "refs/heads/main/quick_start_demo/"
    "chunked_databricks_docs_filtered.jsonl"
)

documents = pd.read_json(docs_url, lines=True)

required_columns = {"content", "doc_uri"}
missing_columns = required_columns.difference(documents.columns)
if missing_columns:
    raise ValueError(
        f"Documentation dataset is missing required columns: {sorted(missing_columns)}"
    )

documents = documents.dropna(subset=["content"]).reset_index(drop=True)
if documents.empty:
    raise ValueError("The documentation dataset contains no usable content.")

doc_vectorizer = TfidfVectorizer(stop_words="english")
tfidf_matrix = doc_vectorizer.fit_transform(documents["content"].astype(str))

print(f"Loaded {len(documents):,} documentation chunks.")

In [ ]:
@tool
@mlflow.trace(
    name="LittleIndex",
    span_type=mlflow.entities.SpanType.RETRIEVER
)
def find_relevant_documents(
    query: str,
    top_n: int = 5
) -> list[dict[str, Any]]:
    """Retrieve the most relevant Databricks documentation chunks."""

    if not query.strip():
        return []

    query_vector = doc_vectorizer.transform([query])
    similarities = (tfidf_matrix @ query_vector.T).toarray().ravel()

    top_n = max(1, min(int(top_n), 10))
    ranked_indices = similarities.argsort()[::-1][:top_n]

    matches = []
    for index in ranked_indices:
        score = float(similarities[index])
        if score <= 0:
            continue

        row = documents.iloc[int(index)]
        matches.append({
            "page_content": str(row["content"]),
            "metadata": {
                "doc_uri": str(row["doc_uri"]),
                "score": score,
            },
        })

    return matches


def tfidf_keywords(text: str) -> list[str]:
    """Extract up to five keywords from the supplied text using TF-IDF."""

    if not text or not text.strip():
        return []

    vectorizer = TfidfVectorizer(stop_words="english")
    matrix = vectorizer.fit_transform([text])
    scores = matrix.toarray()[0]
    feature_names = vectorizer.get_feature_names_out()

    top_indices = scores.argsort()[::-1][:5]
    return [
        str(feature_names[index])
        for index in top_indices
        if scores[index] > 0
    ]

### 5.1 Register the keyword function in Unity Catalog

The function is created in the catalog and schema configured above. This operation requires the appropriate Unity Catalog privileges. If those privileges are unavailable, the exception handler selects the local tool.

In [ ]:
keyword_tools = []
using_unity_catalog = False

try:
    from databricks_langchain.uc_ai import (
        DatabricksFunctionClient,
        UCFunctionToolkit,
        set_uc_function_client,
    )

    uc_client = DatabricksFunctionClient()
    set_uc_function_client(uc_client)

    uc_client.create_python_function(
        func=tfidf_keywords,
        catalog=catalog,
        schema=schema,
        replace=True,
    )

    keyword_tools = UCFunctionToolkit(
        function_names=[f"{catalog}.{schema}.tfidf_keywords"]
    ).tools

    using_unity_catalog = True
    print("Using Unity Catalog keyword tool.")

except Exception as exc:
    print(
        f"Unity Catalog keyword tool unavailable "
        f"({type(exc).__name__}: {exc})."
    )
    print("Falling back to the local LangChain keyword tool.")
    keyword_tools = [tool(tfidf_keywords)]

tools = keyword_tools + [find_relevant_documents]

print("\\nAvailable agent tools:")
for available_tool in tools:
    print(f"- {available_tool.name}")

## 6. Create the LangGraph agent

The agent uses Azure OpenAI as its LLM and the document and keyword functions as tools. MLflow autologging is enabled for tracing.

In [ ]:
import mlflow
from langgraph.prebuilt import create_react_agent

mlflow.langchain.autolog(
    log_traces=True,
    silent=True,
)

SYSTEM_PROMPT = """
You are a helpful assistant that answers questions about Databricks.

Rules:
1. Answer questions about Databricks only.
2. Politely decline unrelated questions without using tools.
3. For Databricks documentation questions, use
   find_relevant_documents and ground your answer in the retrieved content.
4. When the user asks to extract keywords from supplied text, use the
   keyword extraction tool.
5. Do not invent documentation, URLs, or unsupported facts.
6. If the request is unclear, ask a follow-up question.
"""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)


def ask(question: str):
    """Run the agent and return its final answer and the tools it called."""

    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    messages = result["messages"]
    tool_messages = [
        message for message in messages
        if getattr(message, "type", None) == "tool"
    ]

    tools_used = [
        getattr(message, "name", "unknown")
        for message in tool_messages
    ]

    final_message = messages[-1]
    answer = final_message.content

    if isinstance(answer, list):
        answer = "\\n".join(
            item.get("text", str(item)) if isinstance(item, dict) else str(item)
            for item in answer
        )

    return str(answer), tools_used

### 6.1 Test the agent

Run one keyword extraction query and one documentation query.

In [ ]:
answer, used = ask(
    "What are the keywords for: "
    "'the quick brown fox jumped over the lazy brown dog'?"
)
print("Answer:", answer)
print("Tools used:", used)

print("\\n" + "=" * 80 + "\\n")

answer, used = ask("What is Delta Live Tables (DLT) in Databricks?")
print("Answer:", answer)
print("Tools used:", used)

## 7. Evaluate the agent

The evaluation checks whether the expected tool was called and whether the answer contains expected terms. The evaluation results and aggregate metrics are logged to MLflow.

These are lightweight demonstration checks, not a comprehensive measure of factual correctness. In particular, an unrelated-question test checks whether the agent avoids tool calls.

In [ ]:
eval_set = [
    {
        "q": "What is DLT in Databricks?",
        "expect_tool": "find_relevant_documents",
        "expect_terms": ["pipeline"],
    },
    {
        "q": "What is Unity Catalog?",
        "expect_tool": "find_relevant_documents",
        "expect_terms": ["governance"],
    },
    {
        "q": "What API lets me parallelize operations over rows of a delta table?",
        "expect_tool": "find_relevant_documents",
        "expect_terms": ["spark"],
    },
    {
        "q": (
            "Give me keywords for: "
            "'Delta Lake provides ACID transactions for big data workloads'"
        ),
        "expect_tool": "tfidf_keywords",
        "expect_terms": ["delta"],
    },
    {
        "q": "Who won the last cricket world cup?",
        "expect_tool": None,
        "expect_terms": [],
    },
]

rows = []

with mlflow.start_run(run_name="azure_openai_docs_agent_eval"):
    for item in eval_set:
        answer, used = ask(item["q"])

        if item["expect_tool"] is None:
            tool_ok = len(used) == 0
        else:
            tool_ok = any(
                item["expect_tool"] in name
                for name in used
            )

        terms_ok = all(
            term.lower() in answer.lower()
            for term in item["expect_terms"]
        )

        rows.append({
            "question": item["q"],
            "answer": answer,
            "tools_used": ", ".join(used),
            "tool_ok": bool(tool_ok),
            "terms_ok": bool(terms_ok),
        })

    results = pd.DataFrame(rows)

    tool_accuracy = float(results["tool_ok"].mean())
    answer_term_accuracy = float(results["terms_ok"].mean())

    mlflow.log_metric("tool_accuracy", tool_accuracy)
    mlflow.log_metric("answer_term_accuracy", answer_term_accuracy)

    mlflow.log_table(
        results.astype(str),
        "eval_results.json",
    )

display(results)

print(
    f"Tool accuracy: {tool_accuracy:.0%} | "
    f"Answer-term accuracy: {answer_term_accuracy:.0%}"
)

## 8. Conclusion

Built a Databricks-hosted documentation agent using LangGraph and Azure OpenAI (`gpt-5.4-mini`).

The solution combines:
- Azure OpenAI for language-model inference.
- LangGraph for agent orchestration and tool calling.
- TF-IDF for Databricks documentation retrieval.
- Unity Catalog for registering and invoking the keyword extraction function, with a local LangChain fallback.
- MLflow for tracing, evaluation metrics, and evaluation results.

The notebook runs inference directly against Azure OpenAI. Model registration and a Databricks serving endpoint are not required.